<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent; border-radius: 14px; padding: 18px 22px; margin: 12px 0;
  font-size: 36px; font-weight: 600; color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25); background-clip: padding-box; position: relative;
">
  <div style="position: absolute; inset: 0; padding: 4px; border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: linear-gradient(#fff 0 0) content-box, linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor; mask-composite: exclude; pointer-events: none;"></div>
  <b>🏆 Capstone: Probabilistic Financial Forecasting</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [Project Overview](#-part-i-project-overview)**
- **2. [Setup & Synthetic Market Data](#-part-ii-setup)**
- **3. [Bayesian Structural Time Series for Markets](#-part-iii-bsts)**
- **4. [Risk Quantification with Probabilistic Forecasts](#-part-iv-risk)**
    - 4.1 [Value at Risk (VaR)](#var)
    - 4.2 [Scenario Analysis](#scenario-analysis)
- **5. [Decision Making Under Uncertainty](#-part-v-decisions)**
- **6. [Summary](#-part-vi-summary)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. Project Overview</span>

### Why Probabilistic Forecasting for Finance?

| **Traditional Forecast** | **Probabilistic Forecast** |
|-------------------------|---------------------------|
| "Price will be $150" | "Price will be $150 ± $12 (95% CI)" |
| No risk information | P(loss > 5%) = 12% |
| Single scenario | Distribution of scenarios |
| Overconfident | Calibrated uncertainty |

### This Project
- Build a **probabilistic forecasting pipeline** for financial time series
- Compute **Value at Risk (VaR)** from predictive distributions
- Perform **scenario analysis** using posterior samples
- Make **risk-aware investment decisions**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Synthetic Market Data</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

tfd = tfp.distributions
sts = tfp.sts

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

In [ ]:
# Generate realistic synthetic financial time series
np.random.seed(42)
T = 500  # Trading days (~2 years)

# Geometric Brownian Motion with regime changes
dt = 1/252  # Daily
returns = []
price = 100.0
prices = [price]

# Two regimes: normal and volatile
for i in range(T):
    if i < 200:
        mu, sigma = 0.08, 0.15  # Bull market
    elif i < 350:
        mu, sigma = -0.02, 0.35  # Volatile/bear
    else:
        mu, sigma = 0.05, 0.18  # Recovery
    
    daily_return = (mu * dt + sigma * np.sqrt(dt) * np.random.randn())
    price *= (1 + daily_return)
    prices.append(price)
    returns.append(daily_return)

prices = np.array(prices[1:], dtype=np.float32)
returns = np.array(returns, dtype=np.float32)

# Split
train_size = 400
prices_train = prices[:train_size]
prices_test = prices[train_size:]

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
axes[0].plot(prices, color='#3b82f6', linewidth=1.5)
axes[0].axvline(x=train_size, color='red', linestyle='--', alpha=0.7, label='Forecast start')
axes[0].set_ylabel('Price', fontsize=13)
axes[0].set_title('Synthetic Asset Price (with regime changes)', fontsize=15, fontweight='bold')
axes[0].legend()

axes[1].bar(range(T), returns, color=np.where(returns >= 0, '#22c55e', '#ef4444'), alpha=0.7)
axes[1].set_xlabel('Trading Day', fontsize=13)
axes[1].set_ylabel('Daily Return', fontsize=13)
axes[1].set_title('Daily Returns', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. Bayesian Structural Time Series for Markets</span>

In [ ]:
# Model log-returns with STS
returns_train = returns[:train_size]
observed = tf.constant(returns_train[..., tf.newaxis])

# Build STS model for returns
model = sts.Sum([
    sts.LocalLinearTrend(observed_time_series=observed, name='trend'),
    sts.Autoregressive(order=5, observed_time_series=observed, name='ar5'),
], observed_time_series=observed)

# Fit via VI
variational_posteriors = tfp.sts.build_factored_surrogate_posterior(model=model)

@tf.function
def train_vi():
    return tfp.vi.fit_surrogate_posterior(
        target_log_prob_fn=model.joint_distribution(
            observed_time_series=observed).log_prob,
        surrogate_posterior=variational_posteriors,
        optimizer=tf.optimizers.Adam(0.1),
        num_steps=200
    )

losses = train_vi()
print(f"VI training complete. Final ELBO: {-losses.numpy()[-1]:.2f}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Risk Quantification with Probabilistic Forecasts</span>

In [ ]:
# Forecast returns
num_forecast = len(prices_test)
posterior_samples = variational_posteriors.sample(200)

forecast_dist = tfp.sts.forecast(
    model=model,
    observed_time_series=observed,
    parameter_samples=posterior_samples,
    num_steps_forecast=num_forecast
)

# Sample forecast paths
forecast_returns = forecast_dist.sample(500).numpy()[:, :, :, 0]  # (500, 200, T, 1)

# Convert return forecasts to price forecasts
last_price = prices_train[-1]
price_paths = []
for path_idx in range(min(200, forecast_returns.shape[0])):
    # Average over posterior samples dimension
    returns_path = forecast_returns[path_idx].mean(axis=0)
    cum_returns = np.cumprod(1 + returns_path)
    price_paths.append(last_price * cum_returns)
price_paths = np.array(price_paths)

# Statistics
price_mean = price_paths.mean(axis=0)
price_p5 = np.percentile(price_paths, 5, axis=0)
price_p25 = np.percentile(price_paths, 25, axis=0)
price_p75 = np.percentile(price_paths, 75, axis=0)
price_p95 = np.percentile(price_paths, 95, axis=0)

# Plot
fig, ax = plt.subplots(figsize=(16, 8))
t_train = np.arange(train_size)
t_test = np.arange(train_size, T)

ax.plot(t_train, prices_train, color='#3b82f6', linewidth=1.5, label='Historical')
ax.plot(t_test, prices_test, color='#1e293b', linewidth=2, label='Actual')

# Fan chart
ax.fill_between(t_test, price_p5, price_p95, alpha=0.1, color='#ef4444', label='90% CI')
ax.fill_between(t_test, price_p25, price_p75, alpha=0.2, color='#ef4444', label='50% CI')
ax.plot(t_test, price_mean, color='#ef4444', linewidth=2, label='Forecast mean')

# Sample paths
for i in range(20):
    ax.plot(t_test, price_paths[i], color='#8b5cf6', alpha=0.05, linewidth=0.5)

ax.axvline(x=train_size, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Trading Day', fontsize=14)
ax.set_ylabel('Price', fontsize=14)
ax.set_title('Probabilistic Price Forecast (Fan Chart)', fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Value at Risk (VaR) calculation
horizons = [5, 10, 20, 60]  # Trading days

print("="*60)
print("📊 VALUE AT RISK (VaR) ANALYSIS")
print("="*60)
print(f"Current price: ${last_price:.2f}")
print(f"Investment: $10,000\n")

for h in horizons:
    if h <= len(price_paths[0]):
        future_prices = price_paths[:, h-1]
        future_returns = (future_prices - last_price) / last_price
        
        var_95 = np.percentile(future_returns, 5)  # 5th percentile = 95% VaR
        var_99 = np.percentile(future_returns, 1)  # 1st percentile = 99% VaR
        expected_return = np.mean(future_returns)
        prob_loss = np.mean(future_returns < 0)
        
        print(f"{h}-Day Horizon:")
        print(f"  Expected return: {expected_return:+.2%}")
        print(f"  P(loss): {prob_loss:.1%}")
        print(f"  VaR 95%: {var_95:+.2%} (${10000 * abs(var_95):,.0f} at risk)")
        print(f"  VaR 99%: {var_99:+.2%} (${10000 * abs(var_99):,.0f} at risk)")
        print()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Decision Making Under Uncertainty</span>

In [ ]:
# Risk-return analysis
horizon = 20  # 1 month
future_returns = (price_paths[:, min(horizon-1, len(price_paths[0])-1)] - last_price) / last_price

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Return distribution
axes[0].hist(future_returns * 100, bins=40, density=True, alpha=0.7, color='#3b82f6', edgecolor='white')
axes[0].axvline(x=0, color='red', linewidth=2, linestyle='--', label='Break even')
axes[0].axvline(x=np.mean(future_returns)*100, color='#22c55e', linewidth=2, label=f'E[return]={np.mean(future_returns):.1%}')
axes[0].axvline(x=np.percentile(future_returns, 5)*100, color='#ef4444', linewidth=2, 
                label=f'VaR 95%={np.percentile(future_returns, 5):.1%}')
axes[0].set_xlabel(f'{horizon}-Day Return (%)', fontsize=13)
axes[0].set_ylabel('Density', fontsize=13)
axes[0].set_title(f'Predictive Return Distribution ({horizon}-day)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Risk-reward scatter for different investment sizes
fractions = np.linspace(0.1, 1.0, 10)
investment = 10000
expected_gains = []
var_losses = []

for f in fractions:
    gains = investment * f * future_returns
    expected_gains.append(np.mean(gains))
    var_losses.append(-np.percentile(gains, 5))

axes[1].scatter(var_losses, expected_gains, c=fractions, cmap='viridis', s=100, zorder=5, edgecolors='white')
for i, f in enumerate(fractions):
    axes[1].annotate(f'{f:.0%}', (var_losses[i]+5, expected_gains[i]+2), fontsize=9)
axes[1].set_xlabel('95% VaR Loss ($)', fontsize=13)
axes[1].set_ylabel('Expected Gain ($)', fontsize=13)
axes[1].set_title('Risk-Return Trade-off by Position Size', fontsize=14, fontweight='bold')
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('Fraction Invested', fontsize=12)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Summary</span>

### What We Built
- **Bayesian STS model** for financial return forecasting with `tfp.sts`
- **Fan chart** showing predictive uncertainty over multiple horizons
- **Value at Risk (VaR)** directly from the predictive distribution
- **Risk-return trade-off** analysis for position sizing

### Key Techniques Used
| **Technique** | **Purpose** |
|--------------|------------|
| `tfp.sts.Sum` | Decompose returns into trend + autoregressive |
| Variational Inference | Fast approximate posterior inference |
| Forecast sampling | Monte Carlo simulation of future price paths |
| Percentile analysis | VaR and risk metrics from predictive distribution |

### 🎯 This Pattern Applies To
- Portfolio risk management
- Insurance loss modeling
- Credit default prediction
- Macroeconomic forecasting
- Any decision under uncertainty